# Week 5, Day 5: Linear decoding, overfitting and ridge regression

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w005/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def make_trials(n, N=95, rng=rng):
    """Each neuron's rate is weakly tuned to the stimulus s, plus independent noise."""
    s = rng.normal(size=n)
    tuning = np.linspace(-0.4, 0.4, N)
    R = 10 + np.outer(s, tuning) + rng.normal(0, 1, (n, N))
    return R, s

R_train, s_train = make_trials(100)
R_test, s_test = make_trials(1000)
def r2(s, s_hat):
    return 1 - np.sum((s - s_hat) ** 2) / np.sum((s - s.mean()) ** 2)
print(R_train.shape, R_test.shape)

### Exercise: Watch the decoder overfit

For each population size in `sizes`, fit least squares using the first `k` neurons (plus an intercept) on the training trials, and record R² on the training and the test trials. Plot both against `k`. At which size is the decoder best on new trials?

In [ ]:
sizes = [5, 10, 20, 40, 60, 80, 95]
train_r2, new_r2 = [], []
for k in sizes:
    Xtr = np.column_stack([np.ones(len(s_train)), R_train[:, :k]])
    Xte = np.column_stack([np.ones(len(s_test)), R_test[:, :k]])
    beta = ...
    train_r2.append(...)
    new_r2.append(...)

plt.figure(figsize=(5, 3))
plt.plot(sizes, train_r2, "o-", label="training trials"); plt.plot(sizes, new_r2, "o-", label="new trials")
plt.axhline(0, color="0.6", lw=0.8); plt.xlabel("neurons used"); plt.ylabel("R2"); plt.legend(frameon=False); plt.show()
print(np.round(train_r2, 2), np.round(new_r2, 2))

In [ ]:
import numpy as np
def test_lengths():
    "one train and one test R2 per population size"
    assert len(train_r2) == len(sizes) == len(new_r2)

def test_overfit():
    "train R2 rises towards 1 while test R2 falls for the largest population"
    assert train_r2[-1] > 0.95
    assert new_r2[-1] < max(new_r2) - 0.2, f"test R2: {np.round(new_r2, 2)}"

test_lengths()
test_overfit()
print("All checks passed")

### Exercise: Ridge from the formula

Write `ridge(X, y, lam)` that solves $(X^\top X + \lambda I)\boldsymbol\beta = X^\top \mathbf{y}$ with `np.linalg.solve`. For simplicity it penalises every column, so centre the data before using it on real problems (next exercise).

In [ ]:
def ridge(X, y, lam):
    ...

print(np.round(ridge(np.eye(3), np.array([1.0, 2.0, 3.0]), 1.0), 3))

In [ ]:
import numpy as np
def test_zero():
    "ridge with lambda 0 matches least squares"
    X = np.random.default_rng(1).normal(size=(50, 5)); y = np.random.default_rng(2).normal(size=50)
    assert np.allclose(ridge(X, y, 0.0), np.linalg.lstsq(X, y, rcond=None)[0])

def test_shrink():
    "larger penalties give smaller weights"
    X = np.random.default_rng(3).normal(size=(30, 8)); y = np.random.default_rng(4).normal(size=30)
    norms = [np.linalg.norm(ridge(X, y, lam)) for lam in [0.1, 1, 10, 100]]
    assert all(b < a for a, b in zip(norms, norms[1:]))

def test_singular():
    "ridge works even with more neurons than trials"
    X = np.random.default_rng(5).normal(size=(20, 50)); y = np.random.default_rng(6).normal(size=20)
    b = ridge(X, y, 1.0)
    assert b.shape == (50,) and np.all(np.isfinite(b))

test_zero()
test_shrink()
test_singular()
print("All checks passed")

### Exercise: Choose the penalty by cross-validation

Using all 95 neurons and only the training trials, run 5-fold cross-validation for each penalty in `lams`. Within each fold centre with the fold's training means, fit `ridge`, predict the held-out fold, and score R². Store the mean scores in `cv_r2`, the best penalty in `best_lam`, then refit on all training trials and store the test R² in `final_test_r2`.

In [ ]:
def fit_predict(Rtr, s_tr, Rte, lam):
    mu_R, mu_s = Rtr.mean(axis=0), s_tr.mean()
    beta = ridge(Rtr - mu_R, s_tr - mu_s, lam)
    return (Rte - mu_R) @ beta + mu_s

lams = [0.1, 1, 10, 30, 100, 300, 1000, 3000]
folds = np.array_split(rng.permutation(len(s_train)), 5)
cv_r2 = []
for lam in lams:
    scores = []
    for f in folds:
        tr = np.setdiff1d(np.arange(len(s_train)), f)
        ...
    cv_r2.append(np.mean(scores))
best_lam = ...
final_test_r2 = ...

plt.figure(figsize=(5, 3)); plt.semilogx(lams, cv_r2, "o-"); plt.axvline(best_lam, ls="--", color="0.5")
plt.xlabel("ridge penalty λ"); plt.ylabel("cross-validated R2"); plt.show()
print("best lambda:", best_lam, " test R2 with ridge:", round(final_test_r2, 3), " with least squares:", round(new_r2[-1], 3))

In [ ]:
import numpy as np
def test_cv_scores():
    "cv_r2 has one mean cross-validated R2 per lambda"
    assert len(cv_r2) == len(lams)

def test_choice():
    "best_lam is the lambda with the highest cross-validated R2"
    assert best_lam == lams[int(np.argmax(cv_r2))]
    assert best_lam > 0

def test_final():
    "the ridge decoder beats least squares on the test trials"
    assert final_test_r2 > new_r2[-1] + 0.2, f"ridge {final_test_r2:.2f} vs least squares {new_r2[-1]:.2f}"

test_cv_scores()
test_choice()
test_final()
print("All checks passed")